In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import time
import transformers
import torch 
import torch.nn as nn
import os
import ast
from tqdm import tqdm
from transformers import BertModel
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from imblearn.over_sampling import RandomOverSampler

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
d_items = pd.read_csv('path/to/MIMIC/icu/d_items.csv.gz') # < --- Update the path to your d_items.csv.gz file

d_items=d_items[['itemid','label']]

procedureevents = pd.read_csv('path/to/MIMIC/icu/procedureevents.csv.gz') # < --- Update the path to your procedureevents.csv.gz file
procedureevents = procedureevents[['subject_id', 'hadm_id', 'starttime', 'itemid', 'value', 'valueuom']]
procedureevents=procedureevents.merge(d_items, on='itemid', how='inner')
procedureevents = procedureevents.drop(procedureevents[procedureevents['valueuom'] == 'None'].index)

unimportant_events = [
    'ECMO Inflow Line', 'ECMO Outflow Line', 'Subdural Drain',
    'Tandem Heart Outflow Line', 'Tandem Heart Inflow Line', 
    'Centrimag Outflow Line', 'Centrimag Inflow Line', 
    'Multi Lumen Cooling Catheter', 'ICP - PbtO2', 'EVD #2',
    'EKOS', 'SEPS (Subdural Evacuating Port System)', 'ICP - Camino', 
    'Avalon/Protek Duo', '24 Gauge', '14 Gauge', '16 Gauge', 
    '18 Gauge', '20 Gauge', '22 Gauge'
] #dropping events with very low frequency (<100, and those thata re used in routine care and less predictive of LOS)

procedureevents = procedureevents[~procedureevents['label'].isin(unimportant_events)]

procedureevents['code']= procedureevents['label']
procedureevents = procedureevents.rename(columns={'starttime': 'date'})
procedureevents['date'] = pd.to_datetime(procedureevents['date'])
procedureevents['code'] = procedureevents['code'].astype(str)
procedureevents= procedureevents[['subject_id', 'hadm_id', 'date', 'code']]
procedureevents= procedureevents.dropna(subset=['code'])
procedureevents = procedureevents.drop_duplicates()
procedureevents

In [ ]:
chartevents = pd.read_csv('Path/to/MIMIC/icu/datetimeevents.csv.gz') # < --- Update the path to your datetimeevents.csv.gz file


In [ ]:
chartevents.head()

In [ ]:
events=chartevents.merge(d_items, on='itemid', how='inner')
events['warning'] = events['warning'].map({0.0: 'normal', 1.0: 'abnormal'})


columns_to_remove = [
    'Pain Level Acceptable',
    'Oral Care',
    'Skin Integrity',
    'Daily Wake Up',
    'Food and Fluid',
    'Safety Measures',
    'Pain Cause',
    'Plan-Nutritional',
    'Pressure ulcer #10- Drainage amount',
    'Plan-Neuro',
    'Plan-Abdominal/GI',
    'Position',
    'Education Topic',
    'Turn',
    'Position',
    'Diet Type',
    "Avalon Line Site Appear",
    "Recovery RR - Aerobic Capacity",
    "Coping/Knowledge Deficit  NCP - Goal",
    'Humidification',
    '18 Gauge placed in outside facility',
    'CT #1 Dressing',
    'Humidifier Water Changed',
    'Communication',
    'Stool Management',
    'Bath',
    'Warming Device',
    'Incision #2- Dressing Status',
    'Special diet',
    'Any fear in relationships',
    'Education Readiness/Motivation',
    'Education Existing Knowledge',
    'Eye Care',
    'Dressing Status #3',
    '16 Gauge Site Appear',
    'Legal Guardian',
    'Dressing Status #5',
    '16 Gauge placed in the field',
    'CT #1 Leak'
    
    
]

events['code'] = events['label'] + '_' + events['warning'] 
events = events.rename(columns={'charttime': 'date'})
events = events[['subject_id','hadm_id','code','date']]
events['date'] = pd.to_datetime(events['date'])
events = events.dropna(subset=['code'])
events = events.drop_duplicates()
events

In [ ]:
#admission info
admissions = pd.read_csv('Path/to/MIMIC/icu/icustays.csv.gz') # < --- Update the path to your icustays.csv.gz file

admissions = admissions[['subject_id', 'hadm_id', 'intime', 'outtime']]
admissions = admissions.rename(columns={'intime': 'admission_date', 'outtime': 'discharge_date'})
admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])  
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])
admissions = admissions.drop_duplicates()


In [ ]:
dfs = [events, procedureevents]
merged_df = pd.concat(dfs, ignore_index=True)
merged_df.to_csv('Path/to/saved/merged_mimic_dfs.csv') # <-- Update the path to save your merged_mimic_dfs.csv file

df = admissions.merge(merged_df, on=['hadm_id', 'subject_id'], how='inner')
df = df.drop_duplicates()
df.to_csv('Path/to/saved/processed_mimic.csv') # <-- Update the path to save your processed_mimic.csv file
df

In [ ]:
df = df.drop_duplicates(subset=['hadm_id', 'code'])
df

### Merging diagnosis codes on day 1 with all other codes

In [ ]:
#since no datetime info for diagnoses, we assume admissoons came into the ICU with a diagnosis. Therefore we assign on day 1, no time encoding
df=pd.read_csv('Path/to/saved/processed_mimic_data/processed_mimic.csv') # <-- Update the path to your processed_mimic.csv file
df=df.drop_duplicates()
diag1 = pd.read_csv('/Path/to/MIMIC/hosp/diagnoses_icd.csv.gz') # < --- Update the path to your diagnoses_icd.csv.gz file
diag1 = diag1.drop('icd_version', axis=1)
diag2 = pd.read_csv('/Path/to/MIMIC/hosp/d_icd_diagnoses.csv.gz') # < --- Update the path to your d_icd_diagnoses.csv.gz file
diag2 = diag2.drop('icd_version', axis=1)

diag=diag1.merge(diag2, on=['icd_code'], how='inner')
diag['code']=diag['long_title']
diag['Day']='1'
df2=diag[['subject_id', 'hadm_id','code','Day']]
df2=df2.drop_duplicates()
df2

In [ ]:
df2

In [ ]:

df['admission_date'] = pd.to_datetime(df['admission_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])
df['date'] = pd.to_datetime(df['date'])

mask = (df['date'] >= df['admission_date']) & (df['date'] <= df['discharge_date'])
df = df[mask]

df['Day'] = (df['date'] - df['admission_date']).dt.days + 1

df['Day'] = df['Day'].astype(int)

df = pd.concat([df, df2], ignore_index=True)
df.to_csv('Path/to/saved/events_df.csv', index=False) # <-- Update the path to save your events_df.csv file

df


## positionally encoding with the day number/time of day + timeline generation

In [ ]:
#example of single code/string on a given day for a given admission_id
df=pd.read_csv('Path/to/saved/events_df.csv') # <-- Update the path to your events_df.csv file
df= df.sort_values(by=['hadm_id', 'date', 'code'])
print(df.loc[237281,:])

In [ ]:
#now we convert into timelines + save
import tqdm 

def timelines_pandas_chain(df):
    tqdm.tqdm.pandas(desc="to_timeline")

    start_time = time.time()

    timeline = (
        df.groupby(['subject_id', 'hadm_id', 'Day'])['code']
        .progress_apply(list)
        .unstack('Day')
        .reset_index()
        .fillna(0) # Replacing nans with 0s to signify no events
    )
    end_time = time.time()
    print('Total time: ', end_time - start_time)

    return timeline

timeline = timelines_pandas_chain(df)
timeline.to_csv('Path/to/saved/full_timeline_updated.csv') # <-- Update the path to save your full_timeline_updated.csv file
timeline

## Selecting 1st day per admission + Train/test/val split

In [ ]:
#checking the strings for individual days for specific admissions
filtered_value_1 = timeline.iloc[87][1]
filtered_value_2 = timeline.iloc[8][1] #second admission_id, day 1 codes

parsed_list_1 = ast.literal_eval(filtered_value_1) #making the day just one string of all codes
single_string_1 = ', '.join(parsed_list_1)

parsed_list_2 = ast.literal_eval(filtered_value_2)
single_string_2 = ', '.join(parsed_list_2)


print(f'first admission_id, day1:{single_string_1}')


In [ ]:
# Check the data type of the column to ensure it's a list
print(timeline[1].apply(type).value_counts())

# Apply join if the column contains lists
timeline['concatenated_string_day1'] = timeline[1].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

# Select relevant columns
day1 = timeline[['subject_id', 'hadm_id', 'concatenated_string_day1']]
day1


## Getting LOS Labels + assigning to ids


In [ ]:
import transformers
from datasets import Dataset,load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
icustays = pd.read_csv('Path/to/MIMIC/icu/icustays.csv.gz') # < --- Update the path to your icustays.csv.gz file
icustays

In [ ]:
def categorize_los(los):
    if los <= 3:
        return 0  # Short stay
    else:
        return 1  # Long stay

# Apply the function to create a new column 'label'
icustays['label'] = icustays['los'].apply(categorize_los)
print(icustays['label'].value_counts())

In [ ]:
icustays=icustays[['subject_id','hadm_id','los']]
#icustays=icustays[['hadm_id','los']]
icustays=icustays[(icustays['los']>=1)]
labels=icustays.drop_duplicates()

day1=day1.drop_duplicates()
day1= day1[day1['hadm_id'].isin(labels['hadm_id'])]


labels=day1.merge(labels, on='hadm_id',how='inner')
labels.hadm_id.drop_duplicates()
labels= labels.drop_duplicates(subset='hadm_id', keep='first')
labels



In [ ]:
labels.drop(columns = ['subject_id_y'], inplace = True)

In [ ]:
labels.rename(columns = {'subject_id_x': 'subject_id'}, inplace = True)

In [ ]:
df=labels[['subject_id', 'hadm_id','concatenated_string_day1','los']]
df=df.rename(columns={'concatenated_string_day1':'input_ids'})
def categorize_los(los):
    if los <= 3:
        return 0  # Short stay
    else:
        return 1  # Long stay

# Apply the function to create a new column 'label'
df['label'] = df['los'].apply(categorize_los)
print(df['label'].value_counts())

df

In [ ]:
df.to_csv('Path/to/saved/day1.csv') # <-- Update the path to save your day1.csv file    

In [ ]:
df = pd.read_csv('Path/to/saved/day1.csv') # <-- Update the path to your day1.csv file
df.drop(columns = ['Unnamed: 0'], inplace=True)

In [ ]:
df

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss1 = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 42)
train_val_idx, test_idx = next(gss1.split(df, df['label'], groups = df['subject_id']))

X_train_val = df.iloc[train_val_idx].copy()
X_test = df.iloc[test_idx].copy()

gss2 = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 42) 
train_idx, val_idx = next(gss1.split(X_train_val, X_train_val['label'], groups = X_train_val['subject_id']))

X_train = X_train_val.iloc[train_idx].copy()
X_val = X_train_val.iloc[val_idx].copy()

In [ ]:
from imblearn.over_sampling import RandomOverSampler
import numpy as np
import pandas as pd

y_train = X_train['label'].values
input_ids = X_train['input_ids'].values
hadm_ids = X_train['hadm_id'].values
subject_ids = X_train['subject_id'].values

# Reshape input_ids for ROS
X_train_reshaped = input_ids.reshape(-1, 1)

ros = RandomOverSampler(random_state=42)

X_resampled, y_resampled = ros.fit_resample(X_train_reshaped, y_train)

#  Get the resampled indices
resampled_indices = ros.sample_indices_

# Use SAME indices for everything
df_train = pd.DataFrame({
    'input_ids': input_ids[resampled_indices],
    'label': y_resampled,
    'hadm_id': hadm_ids[resampled_indices],
    'subject_id': subject_ids[resampled_indices]
})

print(df_train.label.value_counts())
df_train

In [ ]:
df_train_resampled = df_train_balanced.copy()
df_train_resampled['label'] = y_train_resampled
df_train_resampled.to_csv('Path/to/saved/splits/df_train_resampled_binary_mimic_ids.csv', index=False) # <-- Path to save train split

# Also fix your val/test construction (you created a MultiIndex by accident)
df_val = X_val.copy()
df_val.to_csv('Path/to/saved/df_val_binary_mimic_ids.csv', index=False) # <-- Path to save val split

df_test = X_test.copy()
df_test.to_csv('Path/to/saved/df_test_binary_mimic_ids.csv', index=False) # <-- Path to test split

In [ ]:
df_train_resampled = pd.read_csv('Path/to/saved/splits/df_train_resampled_binary_mimic_ids.csv') # <-- Update the path to your saved train split
df_val = pd.read_csv('Path/to/saved/df_val_binary_mimic_ids.csv') # <-- Update the path to your saved val split
df_test = pd.read_csv('Path/to/saved/df_test_binary_mimic_ids.csv') # <-- Update the path to your saved test split

In [ ]:

list_of_strings_train=df_train_resampled['input_ids'].tolist()
list_of_strings_train = [str(item) for item in list_of_strings_train] #make sure each item has been converted to a string

list_of_strings_val=df_val['input_ids'].tolist()
list_of_strings_val = [str(item) for item in list_of_strings_val]

list_of_strings_test=df_test['input_ids'].tolist()
list_of_strings_test = [str(item) for item in list_of_strings_test]



list_of_strings_train_labels=df_train_resampled['label'].tolist()
list_of_strings_val_labels=df_val['label'].tolist()
list_of_strings_test_labels=df_test['label'].tolist()

print(len(list_of_strings_train))
print(len(list_of_strings_val))


print(len(list_of_strings_train_labels))
print(len(list_of_strings_val_labels))

In [ ]:
#now we tokenise (use med-bert??) for TRAIN data 
from transformers import AutoTokenizer, AutoModel, BertTokenizer, BertModel
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT", weights_only = False)


train=tokenizer(list_of_strings_train, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
train['label']=list_of_strings_train_labels

torch.save(train, 'path/tokenised/medbert_train_lls_tokenised_binary_mimic.pth') # <-- Update the path to save your tokenised train data

val=tokenizer(list_of_strings_val, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
val['label']=list_of_strings_val_labels

torch.save(val, 'path/tokenised/medbert_val_lls_tokenised_binary_mimic.pth')


test=tokenizer(list_of_strings_test, return_tensors='pt', max_length=512, truncation=True, padding='max_length')
test['label']=list_of_strings_test_labels

torch.save(test, 'path/tokenised/medbert_test_lls_tokenised_binary_mimic.pth') # <-- Update the path to save your tokenised test data



In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset

class CustomDataset(Dataset):
    def __init__(self, encodings): # Constructor
        self.encodings=encodings #allows us to retrieve the values passed by calling the encodings attribute
    def __len__(self):
        return len(self.encodings['input_ids'])
    
    def __getitem__(self, idx):     # getting the data

        sample = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['label'][idx]
        }
        return sample
train = torch.load('Path/tokenised/medbert_train_lls_tokenised_binary_mimic.pth', weights_only = False) # <-- Update the path to your tokenised train data
val = torch.load('Path/tokenised/medbert_val_lls_tokenised_binary_mimic.pth', weights_only = False) # <-- Update the path to your tokenised val data

batch_size = 16 

train_dataset = CustomDataset(train)
train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = CustomDataset(val)
val_data_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "emilyalsentzer/Bio_ClinicalBERT" 
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, weights_only=False)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU.")
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")

model.to(device)

class_weights = torch.tensor([1.0, 3.0]).to(device) 

loss_function = nn.CrossEntropyLoss(weight=class_weights)


In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

optimizer = AdamW(model.parameters(),
                  lr=2e-6,
                  eps=1e-8)

epochs = 25
total_steps = len(train_data_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,       
                 num_warmup_steps=0, num_training_steps=total_steps)


In [ ]:
import csv
import torch
from torch.nn.utils.clip_grad import clip_grad_norm_
from tqdm.notebook import tqdm
import wandb

def train(
    model,
    optimizer,
    scheduler,
    epochs,
    loss_function,
    train_data_loader,
    val_data_loader,
    device,
    clip_value=2.0,
    csv_file=None,
    # early stopping
    patience=3,
    min_delta=0.0,
    # wandb
    use_wandb=True,
    wandb_project="Project Name", # <-- Update with your W&B project name
    wandb_run_name=None,
    wandb_config=None,
    log_every_steps=200,
    best_model_path="path/to/best/model/MEDBERT_best_model_binary_mimic.pth", # <-- Update the path to save your best model
):
    """
    Early stopping criteria:
      - If (best_val_loss - avg_val_loss) <= min_delta, counts as no improvement.
      - Stop when no_improve_count >= patience.
    """

    # --- W&B init ---
    run = None
    if use_wandb:
        run = wandb.init(
            project=wandb_project,
            name=wandb_run_name,
            config=wandb_config or {},
            reinit=True,
        )
 

    best_val_loss = float("inf")
    no_improve_count = 0

    train_loss_per_epoch = []
    val_loss_per_epoch = []

    csv_writer = None
    if csv_file:
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(["Epoch", "Train Loss", "Val Loss"])

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        print("Training...")

        model.train()
        total_loss = 0.0

        for step, batch in enumerate(tqdm(train_data_loader, desc="Training")):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_function(logits, labels)

            loss.backward()
            grad_norm = clip_grad_norm_(model.parameters(), clip_value)

            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

            # per-step logging
            if (step + 1) % log_every_steps == 0:
                lr = optimizer.param_groups[0]["lr"]
                print(f"Step {step + 1}/{len(train_data_loader)}, Loss: {loss.item():.6f}, LR: {lr:.3e}")

                if use_wandb:
                    wandb.log(
                        {
                            "train/loss_step": loss.item(),
                            "train/grad_norm": float(grad_norm) if grad_norm is not None else None,
                            "train/lr": lr,
                            "epoch": epoch + 1,
                            "step": epoch * len(train_data_loader) + (step + 1),
                        }
                    )

        avg_train_loss = total_loss / max(1, len(train_data_loader))
        train_loss_per_epoch.append(avg_train_loss)
        print(f"Average training loss: {avg_train_loss:.6f}")

        print("Validation...")
        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for val_step, val_batch in enumerate(tqdm(val_data_loader, desc="Validation")):
                val_input_ids = val_batch["input_ids"].to(device)
                val_attention_mask = val_batch["attention_mask"].to(device)
                val_labels = val_batch["labels"].to(device)

                val_outputs = model(input_ids=val_input_ids, attention_mask=val_attention_mask)
                val_logits = val_outputs.logits
                val_loss = loss_function(val_logits, val_labels)

                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / max(1, len(val_data_loader))
        val_loss_per_epoch.append(avg_val_loss)
        print(f"Average validation loss: {avg_val_loss:.6f}")

        # epoch-level wandb logging
        if use_wandb:
            wandb.log(
                {
                    "train/loss_epoch": avg_train_loss,
                    "val/loss": avg_val_loss,
                    "epoch": epoch + 1,
                }
            )

        # Save best model + early stopping bookkeeping
        improved = (best_val_loss - avg_val_loss) > min_delta
        if improved:
            best_val_loss = avg_val_loss
            no_improve_count = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Model improved; saved to {best_model_path}")

            if use_wandb:
                wandb.run.summary["best_val_loss"] = best_val_loss
                wandb.run.summary["best_epoch"] = epoch + 1
                # If you want the checkpoint uploaded (optional):
                # wandb.save(best_model_path)
        else:
            no_improve_count += 1
            print(f"No improvement for {no_improve_count}/{patience} epoch(s).")

        if csv_writer:
            csv_writer.writerow([epoch + 1, avg_train_loss, avg_val_loss])

        # EARLY STOP
        if no_improve_count >= patience:
            print(f"Early stopping: no val-loss improvement for {patience} epochs.")
            break

    if use_wandb and run is not None:
        run.finish()

    return model


csv_file_path = "path/to/training_results_MEDBERT_binary_mimic.csv" # <-- Update the path to save your training results
with open(csv_file_path, "w", newline="") as csvfile:
    model = train(
        model, optimizer, scheduler, epochs, loss_function,
        train_data_loader, val_data_loader, device,
        clip_value=2.0,
        csv_file=csvfile,
        patience=5,           # x epochs
        min_delta=1e-4,        # optional threshold
        use_wandb=True,
        wandb_project="Project Name", # <-- Update with your W&B project name
        wandb_run_name="bioclinicalBERT_mimic",
        wandb_config={
            "epochs": epochs,
            "clip_value": 2.0,
            "patience": 5,
            "min_delta": 1e-4,
        },
    )

In [ ]:
test=torch.load("Path/to/tokenised/medbert_test_lls_tokenised_binary_mimic.pth") # <-- Update the path to your tokenised test data

model_name = "emilyalsentzer/Bio_ClinicalBERT"  # Confirm the correct model name on Hugging Face
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, weights_only = False)  # Adjust num_labels if needed

model.load_state_dict(torch.load("Path/to/best/model/MEDBERT_best_model_binary_mimic.pth")) # <-- Update the path to your saved best model

if torch.cuda.is_available():       
    device = torch.device("cuda")
    print("Using GPU.")
else:
    print("No GPU available, using the CPU instead.")
    device = torch.device("cpu")

model.to(device)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, encodings): # Constructor
        self.encodings=encodings #allows us to retrieve the values passed by calling the encodings attribute
    def __len__(self):
        return len(self.encodings['input_ids'])
    
    def __getitem__(self, idx):     # getting the data

        sample = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.encodings['label'][idx]
        }
        return sample
    
batch_size = 16 

test_dataset = CustomDataset(test)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report

num_classes = 2
all_probabilities = []
all_predictions = []
all_labels = []

model.eval()

test_loss = 0

with torch.no_grad():
    for step_num, batch_data in enumerate(test_data_loader):
        input_ids = batch_data['input_ids'].to(device)
        attention_mask = batch_data['attention_mask'].to(device)
        labels = batch_data['labels'].to(device)
            
        optimizer.zero_grad()
        output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        logits = output.logits
        loss = loss_function(logits, labels)
        test_loss += loss.item()

        probabilities = F.softmax(logits, dim=-1)
        
        all_probabilities.extend(probabilities.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        predictions = torch.argmax(probabilities, dim=1)
        all_predictions.extend(predictions.cpu().numpy())

accuracy = accuracy_score(all_labels, all_predictions)
print(f'Accuracy: {accuracy}')

report = classification_report(all_labels, all_predictions, target_names=[f'Class {i}' for i in range(num_classes)])
print(report)


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import KFold

num_classes = 2
k_folds = 5

results = []

kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

indices = np.arange(len(test_data_loader.dataset))

for fold, (train_idx, test_idx) in enumerate(kf.split(indices)):
    print(f'Fold {fold+1}/{k_folds}')
    
    train_sampler = torch.utils.data.SubsetRandomSampler(train_idx)
    test_sampler = torch.utils.data.SubsetRandomSampler(test_idx)
    
    train_loader = torch.utils.data.DataLoader(
        test_data_loader.dataset,
        sampler=train_sampler,
        batch_size=test_data_loader.batch_size
    )
    
    test_loader = torch.utils.data.DataLoader(
        test_data_loader.dataset,
        sampler=test_sampler,
        batch_size=test_data_loader.batch_size
    )
    
    model.eval()
    
    all_probabilities = [] 
    all_predictions = []
    all_labels = []
    test_loss = 0
    
    with torch.no_grad():
        for step_num, batch_data in enumerate(test_loader):
            
            input_ids = batch_data['input_ids'].to(device)
            attention_mask = batch_data['attention_mask'].to(device)
            labels = batch_data['labels'].to(device)
            
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            logits = output.logits
            loss = loss_function(logits, labels)
            test_loss += loss.item()

            probabilities = F.softmax(logits, dim=-1)

            predictions = torch.argmax(probabilities, dim=1)

            all_probabilities.extend(probabilities.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            
    accuracy = accuracy_score(all_labels, all_predictions)
    
    report = classification_report(
        all_labels,
        all_predictions,
        target_names=[f'Class {i}' for i in range(num_classes)],
        output_dict=True
    )
        
    fold_result = {
        'Fold': fold+1, 
        'accuracy': accuracy, 
        'short stay precision': report['Class 0']['precision'],
        'short stay recall': report['Class 0']['recall'],
        'short stay f1': report['Class 0']['f1-score'],
        'long stay precision': report['Class 1']['precision'],
        'long stay recall': report['Class 1']['recall'],
        'long stay f1': report['Class 1']['f1-score'],
        'weighted stay precision': report['weighted avg']['precision'],
        'weighted stay recall': report['weighted avg']['recall'],
        'weighted stay f1': report['weighted avg']['f1-score']
    }
    
    results.append(fold_result)
    
results_df = pd.DataFrame(results)
metrics_df = results_df.drop(columns='Fold').agg(['mean', 'std'])

In [ ]:
results_df

In [ ]:
metrics_df